# M1 4단계 — Quick, Draw! 그림 모델 본 학습 (Colab)

위에서부터 한 칸씩 실행한다(각 칸 왼쪽 ▶). 결과는 모두 구글 드라이브 `내 드라이브/hawe-m1/` 에 저장되므로 **연결이 끊겨도 1·2·3번 칸을 다시 실행한 뒤 이어서** 하면 된다.

0. 먼저 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 로 바꾼다.

In [ ]:
# 1. GPU 확인 — 'GPU 0: Tesla T4' 같은 줄이 나와야 한다. 안 나오면 0번(런타임 유형)을 다시 확인
!nvidia-smi -L

In [ ]:
# 2. 구글 드라이브 연결 (팝업에서 계정 선택·허용)
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/hawe-m1'
import os
os.makedirs(WORK, exist_ok=True)
print('작업 폴더:', WORK)

In [ ]:
# 3. 코드 올리기 — 처음 한 번만 파일 선택 창이 뜬다: PC 의 ml/quickdraw/dist/quickdraw_code.zip 을 고른다
import zipfile, io
CODE = f'{WORK}/code'
if not os.path.exists(f'{CODE}/train.py'):
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))).extractall(CODE)
%cd {CODE}
!ls
import tensorflow as tf, keras, numpy, PIL
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| NumPy', numpy.__version__, '| Pillow', PIL.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 4. 데이터 받기 — 50개 범주 × 30,000장 (약 10~20분). 이미 받은 범주는 건너뛴다
!python download.py --out {WORK}/data

In [ ]:
# 5. 학습 — 처음엔 그림 150만 장을 28×28 로 바꾸는 데 20~30분, 그다음 학습 20~45분.
#    바꾼 결과(data/cache)는 드라이브에 남아서 다시 실행하면 바로 학습부터 한다.
!python train.py --data {WORK}/data --runs {WORK}/runs --name full --workers 2

In [ ]:
# 6. 결과 요약 — 이 칸의 출력을 복사해 Claude 에게 붙여 넣으면 된다
import json
m = json.load(open(f'{WORK}/runs/full/metrics.json', encoding='utf-8'))
print(f"범주 {m['classes']}개 | 학습 {m['train_count']:,} / 검증 {m['val_count']:,} / 테스트 {m['test_count']:,}장 | 파라미터 {m['parameters']:,}")
print(f"테스트 top-1 {m['test_top1']:.4f} | top-3 {m['test_top3']:.4f} | 준비 {m['seconds']['prepare']}s | 학습 {m['seconds']['train']}s")
print('버전', m['versions'])
worst = sorted(m['per_class'].items(), key=lambda kv: kv[1]['top1'])[:10]
print('가장 못 맞힌 범주 10개:', ', '.join(f"{k} {v['top1']:.2f}" for k, v in worst))
!ls -la {WORK}/runs/full

In [ ]:
# 7. 크기 줄이기·앱용 파일 (5단계, 약 5분) — 처음 한 번만 파일 선택 창이 뜬다: PC 의 ml/quickdraw/dist/quickdraw_code.zip 을 다시 고른다
#    (export.py 와 golden/ 이 추가된 새 zip. 같은 이름의 파일은 같은 내용으로 덮어쓴다)
if not os.path.exists(f'{CODE}/export.py'):
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))).extractall(CODE)
%cd {CODE}
!python export.py --run {WORK}/runs/full --data {WORK}/data --out {WORK}/export
!ls -la {WORK}/export

8. 다 끝나면 구글 드라이브에서 두 폴더를 PC 로 내려받는다.

- `내 드라이브/hawe-m1/export/` → `ml/quickdraw/export/` (앱에 넣을 모델·라벨·설명서, 깃에 올림)
- `내 드라이브/hawe-m1/runs/full/` → `ml/quickdraw/runs/full/` (학습 결과, 깃 제외)